### Configuring Project Path

This cell ensures the notebook can locate the project's modules by adding the project root directory to Python’s import path (`sys.path`). This setup is required for relative imports from the main application (`app`) to work correctly within the notebook environment.


In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent.parent.parent
sys.path.insert(0, str(PROJECT_ROOT))

### Setup and Imports



In [2]:
from pathlib import Path

DB_PATH = Path("experiments/codecritic.sqlite3")
DB_PATH.parent.mkdir(parents=True, exist_ok=True)

print(f"Ensured directory exists: {DB_PATH.parent}")


Ensured directory exists: experiments


In [3]:
# Cell 1: Imports and Path setup

import sqlite3
from pathlib import Path
from sqlalchemy.orm import Session
from app.db.connection import get_connection, close_connection
from app.db.seeders.seed_prompts import seed_prompts

# Explicitly define your database path
DB_PATH = Path("experiments/codecritic.sqlite3")

print("Setup complete. Ready to proceed.")


Setup complete. Ready to proceed.


## 🔧 Database Initialization

In this step, we establish a connection to the SQLite database and explicitly initialize it, creating tables as defined in our SQLAlchemy models.

This step ensures that the database schema matches our SQLAlchemy definitions.


In [4]:
# Cell 3: Database initialization using SQLAlchemy models

from pathlib import Path
from sqlalchemy import create_engine
from app.db.base import Base

# Ensure directory exists explicitly
DB_PATH = Path("experiments/codecritic.sqlite3")
DB_PATH.parent.mkdir(parents=True, exist_ok=True)

# Create SQLAlchemy engine explicitly
engine = create_engine(f"sqlite:///{DB_PATH}")

# Create tables explicitly
Base.metadata.create_all(bind=engine)

print(f"Database successfully initialized at {DB_PATH.resolve()}")


Database successfully initialized at C:\Repos\codecritic\tests\notebooks\functional_testing_phase\experiments\codecritic.sqlite3


## 🌱 Seeding the Database

Now, we'll seed our database with the predefined prompt data.  
This process:

- Dynamically generates UUIDs for each prompt.
- Copies prompt files to the `extensions` folder.
- Inserts corresponding records into the database.


In [5]:
# Cell 5: Run the seed script explicitly

# Open a new session to seed the database
session = Session(bind=engine)
seed_prompts(session)

session.close()

print("Database seeded successfully.")


Seeded AgentPrompt GUID: a357c165-2fa7-4db2-906b-38423fd1b297
Seeded SystemPrompt GUID: 94a3d3fd-8430-41c5-9640-d10d25c50a55
Database seeded successfully.


## 📖 Querying Seeded Data

Finally, let's explicitly verify that our data was inserted correctly by querying the `agent_prompt` and `system_prompt` tables.


In [6]:
# Cell 7: Query seeded data

conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()

# Query AgentPrompt
print("Agent Prompts:")
for row in cur.execute("SELECT id, guid, name, artifact_path, tags FROM agent_prompt"):
    print(row)

# Query SystemPrompt
print("\nSystem Prompts:")
for row in cur.execute("SELECT id, guid, name, artifact_path, tags FROM system_prompt"):
    print(row)

conn.close()


Agent Prompts:
(1, 'a357c165-2fa7-4db2-906b-38423fd1b297', 'generate', 'C:\\Repos\\codecritic\\extensions\\a357c165-2fa7-4db2-906b-38423fd1b297.txt', '["linting", "generation"]')

System Prompts:
(1, '94a3d3fd-8430-41c5-9640-d10d25c50a55', 'format', 'C:\\Repos\\codecritic\\extensions\\94a3d3fd-8430-41c5-9640-d10d25c50a55.txt', '["formatting", "default"]')


## 📂 Checking Prompt Files

Let's verify the prompt files were created and stored properly within the `extensions` folder.

In [9]:
# Cell 9: Verify prompt files exist in extensions folder

from pathlib import Path

# Get project root explicitly based on notebook's location
PROJECT_ROOT = Path.cwd().parent.parent.parent

extensions_path = PROJECT_ROOT / "extensions"
print("Checking files explicitly in:", extensions_path.resolve())

if not extensions_path.exists():
    print("Extensions directory does not exist!")
else:
    print("Files in 'extensions/' folder:")
    for file in extensions_path.glob("*.txt"):
        print(file.name)


Checking files explicitly in: C:\Repos\codecritic\extensions
Files in 'extensions/' folder:
94a3d3fd-8430-41c5-9640-d10d25c50a55.txt
a357c165-2fa7-4db2-906b-38423fd1b297.txt
